# Reaction-Diffusion with Krylov Subspace Methods

This notebook demonstrates the use of Krylov subspace-based ETD solvers for large sparse systems.

## Heat Equation with Reaction Term

We solve a reaction-diffusion equation with Dirichlet boundary conditions:

$$u_t = \epsilon u_{xx} - \lambda u + f(u)$$

Discretized with finite differences on a uniform grid:

$$u_t = L u + N(u)$$

where $L = \epsilon D_2 - \lambda I$ is the sparse tridiagonal linear operator (all negative eigenvalues for stability).

## Why Krylov Methods?

For large sparse systems, the standard ETD approach using contour integration requires:
- Computing matrix exponentials via dense matrix operations, or
- Diagonalizing the operator (expensive for large systems)

Krylov subspace methods (KIOPS) compute $\varphi$-function actions directly using only matrix-vector products,
making them ideal for:
- Large sparse matrices
- Matrix-free operators
- High-dimensional problems

# Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
import time

# Krylov-based solvers
from rkstiff.etd4k import ETD4K
from rkstiff.etd35k import ETD35K
from rkstiff.krylov import KrylovConfig

# Standard solvers for comparison
from rkstiff.etd4 import ETD4
from rkstiff.etd35 import ETD35
from rkstiff.solveras import SolverConfig
from rkstiff.etd import ETDConfig

%matplotlib inline

# Problem Setup

Create the spatial grid and sparse finite difference Laplacian matrix.
We use a dissipative linear operator to ensure stability.

In [ ]:
# Grid parameters
N = 100  # Number of interior points
x_left, x_right = 0.0, 1.0
dx = (x_right - x_left) / (N + 1)
x = np.linspace(x_left + dx, x_right - dx, N)  # Interior points

# Physical parameters
epsilon = 0.01  # Diffusion coefficient
lam = 1.0       # Linear decay rate

# Sparse tridiagonal Laplacian with Dirichlet BCs (u=0 at boundaries)
D2 = diags([1, -2, 1], [-1, 0, 1], shape=(N, N)) / dx**2

# Linear operator: L = epsilon * D2 - lambda * I (dissipative - all negative eigenvalues)
L = epsilon * D2 - lam * diags([1], [0], shape=(N, N))

print(f"Grid points: {N}")
print(f"Sparse matrix shape: {L.shape}")
print(f"Number of nonzeros: {L.nnz}")
print(f"Sparsity: {100 * L.nnz / (N*N):.2f}%")

# Check eigenvalue range (all should be negative for stability)
L_dense = L.toarray()
eigvals = np.linalg.eigvalsh(L_dense)
print(f"Eigenvalue range: [{eigvals.min():.2f}, {eigvals.max():.2f}]")

# Nonlinear Function

In [ ]:
def nl_func(u):
    """Cubic source term: u^2 (1 - u)"""
    return u**2 * (1 - u)

# Initial Condition

We use a smooth bump initial condition.

In [ ]:
# Initial condition: Gaussian bump
u0 = np.exp(-50 * (x - 0.5)**2)

plt.figure(figsize=(8, 4))
plt.plot(x, u0, 'b-', linewidth=2)
plt.xlabel('x')
plt.ylabel('u')
plt.title('Initial Condition')
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Krylov Configuration

The `KrylovConfig` class controls the Krylov subspace approximation:
- `m_init`: Initial subspace dimension (default: 10)
- `m_max`: Maximum subspace dimension (default: 128)
- `tol`: Convergence tolerance (default: 1e-10)
- `p`: Incomplete orthogonalization parameter (default: 2)

In [ ]:
# Krylov configuration for the exponential integrators
krylov_config = KrylovConfig(
    m_init=15,   # Initial Krylov dimension
    m_max=50,    # Maximum Krylov dimension
    tol=1e-10    # Convergence tolerance
)

# Solver configuration for adaptive stepping
solver_config = SolverConfig(epsilon=1e-5)

# ETD4K: Constant-Step Solver

The `ETD4K` solver uses fourth-order ETD with Krylov subspace methods.
It requires a fixed time step.

In [ ]:
# Create ETD4K solver
solver_etd4k = ETD4K(L, nl_func, krylov_config=krylov_config)

# Time parameters
t0, tf = 0.0, 5.0
h = 0.02  # Fixed time step

# Evolve
start = time.time()
u_final_etd4k = solver_etd4k.evolve(u0.copy(), t0, tf, h, store_data=True)
elapsed_etd4k = time.time() - start

print(f"ETD4K completed in {elapsed_etd4k:.3f} seconds")
print(f"Number of time steps: {len(solver_etd4k.t)}")

# ETD35K: Adaptive-Step Solver

The `ETD35K` solver combines fifth-order ETD with embedded error estimation
for adaptive time stepping. This is more efficient for problems with varying dynamics.

In [ ]:
# Create ETD35K solver with adaptive stepping
solver_etd35k = ETD35K(
    L, nl_func,
    config=solver_config,
    krylov_config=krylov_config
)

# Evolve with adaptive stepping
start = time.time()
u_final_etd35k = solver_etd35k.evolve(u0.copy(), t0, tf, store_data=True)
elapsed_etd35k = time.time() - start

print(f"ETD35K completed in {elapsed_etd35k:.3f} seconds")
print(f"Number of time steps: {len(solver_etd35k.t)}")

# Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ETD4K results
ax1 = axes[0]
U_etd4k = np.array([u.real for u in solver_etd4k.u])
t_etd4k = np.array(solver_etd4k.t)
T, X = np.meshgrid(t_etd4k, x, indexing='ij')
c1 = ax1.pcolormesh(X, T, U_etd4k, shading='auto', cmap='viridis')
ax1.set_xlabel('x')
ax1.set_ylabel('t')
ax1.set_title(f'ETD4K (constant step, {len(t_etd4k)} steps)')
plt.colorbar(c1, ax=ax1, label='u')

# ETD35K results
ax2 = axes[1]
U_etd35k = np.array([u.real for u in solver_etd35k.u])
t_etd35k = np.array(solver_etd35k.t)
T, X = np.meshgrid(t_etd35k, x, indexing='ij')
c2 = ax2.pcolormesh(X, T, U_etd35k, shading='auto', cmap='viridis')
ax2.set_xlabel('x')
ax2.set_ylabel('t')
ax2.set_title(f'ETD35K (adaptive, {len(t_etd35k)} steps)')
plt.colorbar(c2, ax=ax2, label='u')

plt.tight_layout()

# Compare Final States

In [ ]:
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(x, u0, 'k--', label='Initial', linewidth=1.5)
plt.plot(x, u_final_etd4k.real, 'b-', label='ETD4K', linewidth=2)
plt.plot(x, u_final_etd35k.real, 'r--', label='ETD35K', linewidth=2)
plt.xlabel('x')
plt.ylabel('u')
plt.title('Final State Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
diff = np.abs(u_final_etd4k.real - u_final_etd35k.real)
plt.semilogy(x, diff + 1e-16)
plt.xlabel('x')
plt.ylabel('|ETD4K - ETD35K|')
plt.title(f'Difference (max: {np.max(diff):.2e})')
plt.grid(True, alpha=0.3)

plt.tight_layout()

# Adaptive Step Size History

The adaptive solver adjusts the step size based on local error estimates.

In [ ]:
# Compute step sizes from time array
dt_adaptive = np.diff(t_etd35k)

plt.figure(figsize=(10, 4))
plt.semilogy(t_etd35k[1:], dt_adaptive, 'b.-', markersize=3)
plt.axhline(h, color='r', linestyle='--', label=f'ETD4K fixed step = {h}')
plt.xlabel('t')
plt.ylabel('Step size')
plt.title('ETD35K Adaptive Step Sizes')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

print(f"Step size range: [{dt_adaptive.min():.4f}, {dt_adaptive.max():.4f}]")
print(f"Mean step size: {dt_adaptive.mean():.4f}")

# Matrix-Free Operators

The Krylov solvers also support callable operators, enabling matrix-free computations.
This is useful when the operator is too large to store or when using specialized libraries.

In [ ]:
# Define the operator as a callable (matrix-free)
def L_matvec(u):
    """Apply L = epsilon*D2 - lambda*I without forming the matrix."""
    result = -lam * u  # -lambda * I
    # Interior second derivative: epsilon * (u[i-1] - 2*u[i] + u[i+1]) / dx^2
    result[1:-1] += epsilon * (u[:-2] - 2*u[1:-1] + u[2:]) / dx**2
    # Boundary points (Dirichlet u=0 at boundaries)
    result[0] += epsilon * (-2*u[0] + u[1]) / dx**2
    result[-1] += epsilon * (u[-2] - 2*u[-1]) / dx**2
    return result

# Create solver with callable operator
solver_mf = ETD4K(L_matvec, nl_func, krylov_config=krylov_config)

# Evolve
start = time.time()
u_final_mf = solver_mf.evolve(u0.copy(), t0, tf, h, store_data=False)
elapsed_mf = time.time() - start

print(f"Matrix-free solver completed in {elapsed_mf:.3f} seconds")

# Compare with sparse matrix version
diff_mf = np.abs(u_final_mf.real - u_final_etd4k.real)
print(f"Max difference from sparse version: {np.max(diff_mf):.2e}")

# Summary

The Krylov subspace-based ETD solvers provide:

1. **ETD4K**: Fourth-order accuracy with constant step size
   - Good for problems with uniform time scales
   - Simple to use

2. **ETD35K**: Fifth-order accuracy with adaptive stepping
   - Embedded error estimation for automatic step size control
   - More efficient for problems with varying dynamics

Both solvers:
- Work with sparse matrices
- Support matrix-free operators (callables)
- Scale well with problem size
- Preserve the stability properties of exponential integrators